# V3 Phase 0 — Failure Analysis du baseline 0.764

Setup : `classifier_v4_realmix.pt` (11.2M) + `detector_baseline.pt` (0.53M), hybride + TTA.

Catégorise les erreurs en 6 types : FN_detection, FP_detection, misclassification, wrong_player, center_error, active_error.

Les prédictions sont produites par `scripts/v3_failure_analysis.py` (sauvé dans `reports/v3_failure_analysis.csv`).

In [ ]:
import sys
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
df = pd.read_csv(ROOT / 'reports' / 'v3_failure_analysis.csv')
n = len(df)
print(f'{n} images, {df.n_gt.sum()} cartes GT, {df.n_pred.sum()} cartes predites')
df.head()

## 1. Décomposition des erreurs cartes

In [ ]:
cats = ['FN_detection', 'FP_detection', 'misclassification', 'wrong_player']
totals = {c: int(df[c].sum()) for c in cats}
card_err = sum(totals.values())
for c, v in totals.items():
    print(f'{c:20s}: {v:4d}  ({100*v/card_err:5.1f}%)')
print(f'{"TOTAL":20s}: {card_err:4d} sur {df.n_gt.sum()} GT')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(totals.keys(), totals.values(), color=['#f1c40f','#e67e22','#e74c3c','#2ecc71'])
ax.set_ylabel('count'); ax.set_title('Erreurs cartes par categorie (baseline 0.764)')
for i, v in enumerate(totals.values()):
    ax.text(i, v + 1, str(v), ha='center')
plt.tight_layout(); plt.show()

## 2. Erreurs scène (center / active)

In [ ]:
center_acc = 1 - df.center_error.sum() / n
active_acc = 1 - df.active_error.sum() / n
print(f'CenterAcc : {center_acc:.3f}  ({df.center_error.sum()} err)')
print(f'ActiveAcc : {active_acc:.3f}  ({df.active_error.sum()} err)')

print('\nConfusion CENTER :')
bad = df[df.center_error == 1]
for (g, p), c in Counter(zip(bad.gt_center, bad.pred_center)).most_common(10):
    print(f'  {g:>10} -> {p:>10} : {c}x')

print('\nConfusion ACTIVE :')
bada = df[df.active_error == 1]
for (g, p), c in Counter(zip(bada.gt_active, bada.pred_active)).most_common(10):
    print(f'  {g:>5} -> {p:>5} : {c}x')

## 3. Pires images (à inspecter visuellement)

In [ ]:
df['total_err'] = df[cats].sum(axis=1) + df.center_error + df.active_error
print(df.nlargest(12, 'total_err')[['image_id'] + cats + ['center_error','active_error','n_gt','n_pred']].to_string(index=False))

## 4. Synthèse — où sont les 23.6 % d'erreurs ?

| Catégorie | % erreurs cartes | Levier V3 |
|---|---|---|
| **misclassification** | **58.8 %** | corner-classification (info discriminante dans les coins) |
| FP_detection | 28.6 % | déduplication géométrique 2-coins |
| FN_detection | 9.9 % | redondance 2 coins (robuste occlusion) |
| wrong_player | 2.7 % | déjà OK — ne pas toucher |

**Conclusion** : le levier dominant est la **classification** (59 %). L'approche V3 corner-classification + dédup géométrique attaque directement les 2 premiers leviers (88 % des erreurs). GO conceptuel — validation empirique en Phase 1.